# Notebook 06 — Text Classification with a Transformer

## Learning Objectives
- Load real text data from a CSV file using `src.data.tiny_text`.
- Build a vocabulary and encode sentences as padded integer sequences.
- Train a `TextTransformerClassifier` (positive / negative / neutral sentiment).
- Plot training curves, a confusion matrix, and sample predictions.
- Save all outputs to `runs/text_classification/`.

## Big Picture
Text classification is one of the most common NLP tasks. Given a sentence,
predict its category (sentiment, topic, intent, …). The Transformer encoder
reads the full sentence in parallel, then a linear head maps the pooled
representation to class logits.

## 1. Imports and Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

from src.utils.seed import seed_everything
from src.utils.device import get_best_device
from src.utils.cleanup import prepare_run_dir
from src.utils.reporting import save_metrics_json, save_markdown_report, save_table_csv
from src.utils.paths import RUNS_TEXT_CLS_DIR, TINY_SENTIMENT_CSV
from src.data.tiny_text import (
    load_sentiment_data, IDX_TO_LABEL, LABEL_TO_IDX, tokenize
)
from src.models.text_transformer import TextTransformerClassifier

seed_everything(42)
device = get_best_device()
prepare_run_dir(RUNS_TEXT_CLS_DIR, clean=True)
print(f"Device: {device}")
print(f"Run dir: {RUNS_TEXT_CLS_DIR}")

## 2. Dataset: Tiny Sentiment CSV

In [ ]:
# Load and inspect the raw CSV
df = pd.read_csv(TINY_SENTIMENT_CSV)
print(f"Dataset path  : {TINY_SENTIMENT_CSV}")
print(f"Total rows    : {len(df)}")
print(f"Columns       : {list(df.columns)}")
print(f"\nLabel distribution:")
print(df['label'].value_counts())
print(f"\nFirst 5 samples:")
print(df.head())

In [ ]:
# Hyperparameters
MAX_LEN    = 32
BATCH_SIZE = 16
D_MODEL    = 64
NUM_HEADS  = 4
NUM_LAYERS = 2
DIM_FF     = 128
NUM_EPOCHS = 8
LR         = 5e-4
NUM_CLASSES = 3  # positive, negative, neutral

# Load data using src utility
train_loader, val_loader, vocab = load_sentiment_data(
    csv_path=TINY_SENTIMENT_CSV,
    max_len=MAX_LEN,
    batch_size=BATCH_SIZE,
    num_workers=0,
    seed=42,
)

VOCAB_SIZE = len(vocab)
print(f"Vocabulary size : {VOCAB_SIZE}")
print(f"Train batches   : {len(train_loader)}")
print(f"Val   batches   : {len(val_loader)}")

# Inspect one batch
sample_ids, sample_labels = next(iter(train_loader))
print(f"\nBatch input_ids shape : {sample_ids.shape}   (batch, max_len)")
print(f"Batch labels shape    : {sample_labels.shape}")
print(f"Label values          : {sample_labels.tolist()[:8]}")
print(f"Label names           : {[IDX_TO_LABEL[l.item()] for l in sample_labels[:8]]}")

## 3. Architecture (Text Diagram)

```
Raw text: "I love this movie"
        │
  Tokenize → ['i', 'love', 'this', 'movie']
        │
  Encode  → [word_id, word_id, word_id, word_id, <pad>, ..., <pad>]   length=32
        │
  Token Embedding × √d_model    [batch, 32, 64]
        │
  + Sinusoidal Positional Encoding
        │
  TransformerEncoder (2 blocks × {4 heads, d_model=64, FFN=128})
        │
  Mean Pool (excluding padding)  [batch, 64]
        │
  Linear(64 → 3)                 [batch, 3]
        │
  Argmax → class label: positive / negative / neutral
```

## 4. Theory: Text Classification with a Transformer

**Token embeddings** map discrete word IDs to dense vectors. We scale them by √d_model
(following the original paper) to keep the positional encoding signal comparable.

**Padding mask**: Variable-length sentences are padded to a fixed max_len. We pass a
boolean mask to the encoder so padded positions don't influence the output.

**Mean pooling** averages all non-padded token outputs into a single sentence vector.
This is often called a *sentence embedding*. Alternatively, a CLS token (prepended
special token) can be used — the `TextTransformerClassifier` supports both.

**Cross-entropy loss** is standard for multi-class classification:
$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^N \log p(y_i \mid x_i)$$

## 5. Implementation from Scratch

In [ ]:
# Build the model
model = TextTransformerClassifier(
    vocab_size=VOCAB_SIZE,
    num_classes=NUM_CLASSES,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FF,
    max_len=MAX_LEN + 4,
    dropout=0.1,
    pad_idx=0,
    use_cls_token=False,
).to(device)

n_params = model.count_parameters()
print(f"Trainable parameters: {n_params:,}")

# Trace through the shapes
ids = sample_ids.to(device)
with torch.no_grad():
    logits, _ = model(ids)
print(f"\nInput  shape  : {ids.shape}      (batch, max_len)")
print(f"Output logits : {logits.shape}   (batch, num_classes)")
print(f"Predicted class: {logits.argmax(dim=-1).cpu().tolist()[:8]}")

## 6. Sanity Checks

In [ ]:
# Shape checks
assert logits.shape == (BATCH_SIZE, NUM_CLASSES), f"{logits.shape}"
print("Shape check passed.")

# Probabilities should sum to 1
probs = logits.softmax(dim=-1)
assert torch.allclose(probs.sum(dim=-1), torch.ones(BATCH_SIZE, device=device), atol=1e-5)
print("Softmax sum-to-1 check passed.")

# Initial loss should be close to log(3) ≈ 1.099 (random predictions, 3 classes)
import math
init_loss = nn.CrossEntropyLoss()(logits, sample_labels.to(device)).item()
expected  = math.log(NUM_CLASSES)
print(f"Initial loss: {init_loss:.4f}  (expected ≈ {expected:.4f} = log({NUM_CLASSES}))")
print("All sanity checks passed!")

## 7. Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

train_losses, val_losses = [], []
train_accs,   val_accs   = [], []

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Train ──
    model.train()
    epoch_loss, correct, total = 0.0, 0, 0
    for ids, labels in train_loader:
        ids, labels = ids.to(device), labels.to(device)
        optimizer.zero_grad()
        logits, _ = model(ids)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
        correct += (logits.argmax(dim=-1) == labels).sum().item()
        total   += labels.size(0)
    train_losses.append(epoch_loss / len(train_loader))
    train_accs.append(correct / total)

    # ── Validate ──
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for ids, labels in val_loader:
            ids, labels = ids.to(device), labels.to(device)
            logits, _ = model(ids)
            v_loss += criterion(logits, labels).item()
            v_correct += (logits.argmax(dim=-1) == labels).sum().item()
            v_total   += labels.size(0)
    val_losses.append(v_loss / len(val_loader))
    val_accs.append(v_correct / v_total)

    print(f"Epoch {epoch:2d}/{NUM_EPOCHS}  "
          f"train_loss={train_losses[-1]:.4f}  train_acc={train_accs[-1]:.3f}  "
          f"val_loss={val_losses[-1]:.4f}  val_acc={val_accs[-1]:.3f}")

print("Training complete!")

## 8. Evaluation

In [ ]:
# Collect all predictions for confusion matrix
model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for ids, labels in val_loader:
        ids = ids.to(device)
        logits, _ = model(ids)
        all_preds.extend(logits.argmax(dim=-1).cpu().tolist())
        all_true.extend(labels.tolist())

final_acc = sum(p == l for p, l in zip(all_preds, all_true)) / len(all_true)
print(f"Final validation accuracy: {final_acc:.4f}  ({final_acc*100:.1f}%)")

# Confusion matrix
class_names = ['positive', 'negative', 'neutral']
n_classes   = len(class_names)
conf_matrix = np.zeros((n_classes, n_classes), dtype=int)
for pred, true in zip(all_preds, all_true):
    conf_matrix[true, pred] += 1

print("\nConfusion Matrix (rows=true, cols=pred):")
print(f"{'':>12}" + "".join(f"{c:>12}" for c in class_names))
for i, name in enumerate(class_names):
    print(f"{name:>12}" + "".join(f"{conf_matrix[i,j]:>12}" for j in range(n_classes)))

# Per-class accuracy
print("\nPer-class accuracy:")
for i, name in enumerate(class_names):
    total_i = conf_matrix[i].sum()
    if total_i > 0:
        print(f"  {name:<12}: {conf_matrix[i,i]/total_i:.3f}  ({conf_matrix[i,i]}/{total_i})")

In [ ]:
# Sample predictions
print("\nSample predictions:")
print("-" * 70)
val_df = pd.read_csv(TINY_SENTIMENT_CSV)
# Get a few samples from the val split
model.eval()
sample_texts = []
for ids, labels in val_loader:
    sample_texts.append((ids[:5], labels[:5]))
    break

ids_sample, labels_sample = sample_texts[0]
with torch.no_grad():
    logits_sample, _ = model(ids_sample.to(device))
    probs_sample = logits_sample.softmax(dim=-1).cpu()

# Decode token IDs back to words
idx2word = {v: k for k, v in vocab.items()}
print(f"{'Decoded Input':<35} {'True':>10} {'Predicted':>12} {'Confidence':>12}")
print("-" * 70)
for i in range(min(5, ids_sample.size(0))):
    words = [idx2word.get(int(t), '<unk>') for t in ids_sample[i] if int(t) != 0]
    decoded = ' '.join(words[:7])
    true_lbl = IDX_TO_LABEL[labels_sample[i].item()]
    pred_idx = probs_sample[i].argmax().item()
    pred_lbl = IDX_TO_LABEL[pred_idx]
    conf = probs_sample[i][pred_idx].item()
    print(f"{decoded:<35} {true_lbl:>10} {pred_lbl:>12} {conf:>12.3f}")

## 9. Visualization

In [ ]:
# Figure 1: Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ep = range(1, NUM_EPOCHS + 1)

ax1.plot(ep, train_losses, 'o-', label='Train', color='steelblue')
ax1.plot(ep, val_losses,   's-', label='Val',   color='tomato')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Cross-Entropy Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(ep, [a*100 for a in train_accs], 'o-', label='Train', color='steelblue')
ax2.plot(ep, [a*100 for a in val_accs],   's-', label='Val',   color='tomato')
ax2.axhline(100/NUM_CLASSES, color='gray', linestyle='--', alpha=0.5, label='Random')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy'); ax2.legend(); ax2.grid(True, alpha=0.3)

fig.suptitle('Text Classification — Training Curves', fontsize=12)
fig.tight_layout()
curve_path = RUNS_TEXT_CLS_DIR / 'training_curve.png'
fig.savefig(curve_path, dpi=120)
plt.close(fig)
print(f"Curve saved to: {curve_path}")

In [ ]:
# Figure 2: Confusion matrix heatmap
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(conf_matrix, cmap='Blues')
ax.set_xticks(range(n_classes))
ax.set_yticks(range(n_classes))
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix')
for i in range(n_classes):
    for j in range(n_classes):
        ax.text(j, i, str(conf_matrix[i, j]), ha='center', va='center',
                color='white' if conf_matrix[i,j] > conf_matrix.max()/2 else 'black',
                fontsize=14)
plt.colorbar(im, ax=ax)
fig.tight_layout()
cm_path = RUNS_TEXT_CLS_DIR / 'confusion_matrix.png'
fig.savefig(cm_path, dpi=120)
plt.close(fig)
print(f"Confusion matrix saved to: {cm_path}")

In [ ]:
# Save metrics and session report
metrics = {
    'final_val_accuracy': final_acc,
    'final_val_loss': val_losses[-1],
    'final_train_loss': train_losses[-1],
    'num_epochs': NUM_EPOCHS,
    'vocab_size': VOCAB_SIZE,
    'num_params': n_params,
}
save_metrics_json(metrics, RUNS_TEXT_CLS_DIR / 'metrics.json')

save_markdown_report(
    title='Text Classification',
    summary=f'TextTransformerClassifier trained for {NUM_EPOCHS} epochs on tiny_sentiment.csv. '
            f'Final val accuracy: {final_acc:.4f}.',
    metrics=metrics,
    figures=['training_curve.png', 'confusion_matrix.png'],
    tables=[],
    output_path=RUNS_TEXT_CLS_DIR / 'session_report.md',
    device=str(device),
    hyperparams={'d_model': D_MODEL, 'heads': NUM_HEADS, 'layers': NUM_LAYERS,
                 'lr': LR, 'batch_size': BATCH_SIZE, 'max_len': MAX_LEN},
    architecture='TextTransformerClassifier: embedding + SinusoidalPE + TransformerEncoder × 2 + mean_pool + Linear',
    loss_fn='CrossEntropyLoss',
)
print(f"Session report saved to: {RUNS_TEXT_CLS_DIR / 'session_report.md'}")

## 10. Failure Cases

**Overfitting on tiny data**: With only 90 samples, a large model will memorize training data. Signs: train_acc ≈ 100%, val_acc ≈ 33% (random). Fix: increase dropout, reduce model size, or use data augmentation.

**Unknown words at inference**: The `encode_sentence` function maps OOV tokens to `UNK_IDX`. If test vocabulary is very different from training, accuracy will be poor.

**Class imbalance**: If one class has many more samples, the model may always predict that class. Check `df['label'].value_counts()` — our tiny dataset is balanced.

**Learning rate too high**: Cross-entropy explodes early in training. Reduce LR or use a warmup schedule.

## 11. Exercises

1. **CLS token**: Set `use_cls_token=True` in the model. Does accuracy change? Why might CLS work better or worse than mean pooling on such short sentences?
2. **Dropout ablation**: Try dropout=0.0 and dropout=0.5. Plot both training curves. Which overfits more?
3. **Attention visualization**: After training, run a positive-sentiment sentence through the model with `return_weights=True`. Visualize which words attend to which.
4. **Label smoothing**: Replace `CrossEntropyLoss()` with `CrossEntropyLoss(label_smoothing=0.1)`. Does validation accuracy improve?
5. **Error analysis**: Print all misclassified validation examples. Do you see a pattern? Are neutral sentences harder to classify than positive/negative?

## 12. Key Takeaways

- **TextTransformerClassifier** = embedding + positional encoding + encoder stack + mean pool + linear head.
- **Padding mask** ensures padded tokens don't influence the sentence representation.
- **Scaling embeddings by √d_model** balances the magnitude of token embeddings vs positional encoding.
- On tiny datasets, a small model (d_model=64, 2 layers) outperforms larger models due to overfitting.
- The confusion matrix reveals which classes are most easily confused — often neutral is hardest.